In [104]:
import numpy as np
from sklearn.datasets import fetch_openml
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ExponentialLR, StepLR
import torchvision.transforms.v2 as transforms_v2
from torchvision import datasets
import matplotlib.pyplot as plt

#CONFIGURATION
SEEDS = [1, 3, 5]
BATCH_SIZE = 100
EPOCHS = 100
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NORMALIZATION = "zscore" #"minmax", "zscore" or "None"
INITIALIZATION = "normal" #"he","normal", "uniform" or "None"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CROP_PADDING_NUM = 2
N_FIRST_LAYER = 128
N_SECOND_LAYER = 64
N_THIRD_LAYER = 32
OVERFITTING_DETECTION_PARAMETER = 10

DROPOUT = False
DROPOUT_VALUE = 0.25

#adaptive learning rate
ALR = "exp_decay" #"exp_decay", #"factor_decay" #"None"
GAMMA = 0.95 #rate of exp decay

sigma = 1.0

#percentage of labeled data
LABELED_DATA_PERCENT = 50 #5,10,25,50,100

#Data augmentation
AUGMENTATION = True

augmenter = transforms_v2.Compose([
    transforms_v2.RandomHorizontalFlip(p=0.5),
    transforms_v2.RandomCrop(28, padding=CROP_PADDING_NUM),
])

train_part = datasets.FashionMNIST(root='./data', train=True, download=True)
test_part = datasets.FashionMNIST(root='./data', train=False, download=True)

X_train_raw = train_part.data
Y_train_raw = train_part.targets
X_test_raw = test_part.data
Y_test_raw = test_part.targets

X = torch.cat([X_train_raw, X_test_raw], dim=0)
Y = torch.cat([Y_train_raw, Y_test_raw], dim=0)


In [105]:
#DEFINE NN CLASS
class NeuralNetwork(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = torch.nn.Flatten()
    self.linear_relu_stack = None
    if DROPOUT:
      self.linear_relu_stack = torch.nn.Sequential(
        torch.nn.Linear(28*28,N_FIRST_LAYER),
        torch.nn.ReLU(),
        torch.nn.Linear(N_FIRST_LAYER,N_SECOND_LAYER),
        torch.nn.ReLU(),
        torch.nn.Dropout(DROPOUT_VALUE),
        torch.nn.Linear(N_SECOND_LAYER,N_THIRD_LAYER),
        torch.nn.ReLU(),
        torch.nn.Dropout(DROPOUT_VALUE),
        torch.nn.Linear(N_THIRD_LAYER,10)
      )
    else:
      self.linear_relu_stack = torch.nn.Sequential(
      torch.nn.Linear(28*28,N_FIRST_LAYER),
      torch.nn.ReLU(),
      torch.nn.Linear(N_FIRST_LAYER,N_SECOND_LAYER),
      torch.nn.ReLU(),
      torch.nn.Linear(N_SECOND_LAYER,N_THIRD_LAYER),
      torch.nn.ReLU(),
      torch.nn.Linear(N_THIRD_LAYER,10)
    )

  def forward(self,x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

#INITIALIZATION
"""this function checks if the current layer is a linear layer, if it is,
the weights are changed to He init. m represents current layer"""
def weights_init(m):
  if isinstance(m, torch.nn.Linear):
    if(INITIALIZATION == "he"):
      torch.nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
    elif(INITIALIZATION == "normal"):
      torch.nn.init.normal_(m.weight,mean=0,std=0.05)
    else:
      fanin, __ = torch.nn.init._calculate_fan_in_and_fan_out(m.weight)
      torch.nn.init.uniform_(m.weight,a= -1/np.sqrt(fanin),b=1/np.sqrt(fanin))


#TRACK DATA
results = {
    "train_loss": [],
    "valid_loss": [],
    "train_acc": [],
    "valid_acc": [],
    "test_acc": [],
    "converge_time": [],
}

"""this function gets loss and accuracy with a given dataset and labels"""
def eval_model(x_set,y_set,model,criterion):
  with torch.no_grad(): #this disables gradient calculations
    x_set = x_set.to(DEVICE)
    y_set = y_set.to(DEVICE)

    #run through model and get predictions
    outputs = model(x_set)
    loss = criterion(outputs, y_set)
    probab = torch.nn.Softmax(dim=1)(outputs)
    predictions = probab.argmax(1)

    #take off GPU to use numpy
    numpy_pred = predictions.cpu().detach().numpy()
    y_valid_numpy = y_set.cpu().detach().numpy()

    #calc and return the accuracy and loss
    acc = numpy_pred == y_valid_numpy
    return loss.item(),acc.astype(int).sum()/len(acc)




THIS NOTEBOOK RUNS WHEN USING 100% OF LABELS

In [3]:
#THIS IS THE SIMULATION LOOP FOR USING 100% OF TRAINING DATA AS LABELS
if LABELED_DATA_PERCENT == 100:
  #SIMULATION LOOP
  for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)

    #RANDOMLY PERMUTATE DATASET
    indices = np.random.permutation(len(Y))
    X = X[indices]
    Y = Y[indices]

    #SPLIT DATA INTO SPECIFIED REGIONS
    test_set = X[0:7000] #10% of df, 7k data points
    y_test = Y[0:7000]

    valid_set = X[7000:14000] #10% of df, 7k data points
    y_valid = Y[7000:14000]

    train_set = X[14000:] #80% of df, 56k data points
    y_train = Y[14000:]

    #only need these three sets now!

    #CONVERT DATA SPLITS TO TENSORS
    test = torch.tensor(test_set, dtype=torch.float32).to(DEVICE)
    y_test = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
    valid = torch.tensor(valid_set, dtype=torch.float32).to(DEVICE)
    y_valid = torch.tensor(y_valid, dtype=torch.long).to(DEVICE)
    train = torch.tensor(train_set, dtype=torch.float32).to(DEVICE)
    y_train = torch.tensor(y_train, dtype=torch.long).to(DEVICE)

    #INSTANTIATE MODEL
    model = NeuralNetwork().to(DEVICE)
    if INITIALIZATION != "None":
      model.apply(weights_init)

    #DEFINE CRITERIA AND OPTIMIZER
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE ,momentum = MOMENTUM) #m = 0.5,0.99; lr = 0.001, 0.1
    #optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = None
    if ALR == "exp_decay":
      scheduler = ExponentialLR(optimizer, gamma=GAMMA)
    elif ALR == "factor_decay":
      scheduler = StepLR(optimizer, step_size=25, gamma=0.1)

    #NORMALIZE VALID AND TEST HERE B/C THEY AREN'T BEING AUGMENTED
    if NORMALIZATION == "minmax":
        test = test / 255.0
        valid = valid / 255.0
    elif NORMALIZATION == "zscore":
        # Simple Z-score approximation
        valid = (valid - valid.mean()) / valid.std()
        test = (test - test.mean()) / test.std()



    #DEFINE VARS FOR MINIBATCH GRADIENT DESCENT
    dataset = TensorDataset(train,y_train)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    #DATA COLLECTION FOR THIS SEED
    seed_train_loss, seed_valid_loss = [], []
    seed_train_acc, seed_valid_acc = [], []
    loss_history = np.zeros(OVERFITTING_DETECTION_PARAMETER)
    loss_history[loss_history == 0.0] = np.nan #fix this later

    #FORWARD PASS WITH INITIALIZED WEIGHTS
    for epoch in range(EPOCHS):
      model.train()

      #VARS FOR MINIBATCH STORAGE
      running_mean_loss = 0 #stores average loss over minibatches, so this is mean per epoch
      running_mean_acc = 0 #stores acc loss over minibatches, so this is acc per epoch
      count = 1 #counts minibatches

      #loads minibatch
      for x,y in dataloader:
          x, y = x.to(DEVICE), y.to(DEVICE)

          if AUGMENTATION:
            x = augmenter(x)
            # plt.imshow(x[0])
            # plt.show()

          if NORMALIZATION == "minmax":
              x = x / 255.0
          elif NORMALIZATION == "zscore":
              x = (x - x.mean()) / x.std()

          #forward pass and loss
          outputs = model(x)
          loss = criterion(outputs, y)

          #backprop
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          #get the loss and acc of the current minibatch
          minibatch_loss, minibatch_acc = eval_model(x, y,model,criterion)

          #store average loss and acc of minibatches
          running_mean_loss = running_mean_loss + (minibatch_loss - running_mean_loss) / count
          running_mean_acc = running_mean_acc + (minibatch_acc - running_mean_acc) / count
          count += 1

      #store the minibatch averages as per-epoch averages
      seed_train_loss.append(running_mean_loss)
      seed_train_acc.append(running_mean_acc)

      #display avg. training loss
      if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}, Loss: {running_mean_loss:.4f}')

      #validation
      model.eval()
      valid_loss, valid_acc = eval_model(valid, y_valid,model,criterion)

      seed_valid_loss.append(valid_loss)
      seed_valid_acc.append(valid_acc)
      loss_history[epoch%OVERFITTING_DETECTION_PARAMETER] = valid_loss

      #check overfitting
      mean = np.nanmean(loss_history)
      std = np.nanstd(loss_history)
      if valid_loss > mean + std or epoch == EPOCHS-1:
        print('overfitting',epoch,valid_loss,valid_acc)
        results['converge_time'].append(epoch)
        break

      if ALR == "exp_decay" or ALR == "factor_decay":
        scheduler.step() #decay learning rate
        # current_lr = optimizer.param_groups[0]['lr']
        # print(f"Current Learning Rate: {current_lr}")




    results['train_loss'].append(seed_train_loss)
    results['valid_loss'].append(seed_valid_loss)
    results["train_acc"].append(seed_train_acc)
    results["valid_acc"].append(seed_valid_acc)


    #TESTING
    logits = model(test)
    pred_probab = torch.nn.Softmax(dim=1)(logits)
    y_pred = pred_probab.argmax(1)
    print(f"Predicted class: {y_pred}")

    _,final_acc = eval_model(test,y_test,model,criterion)

    results["test_acc"].append(final_acc)


/tmp/ipython-input-2382427777.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test = torch.tensor(test_set, dtype=torch.float32).to(DEVICE)
/tmp/ipython-input-2382427777.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
/tmp/ipython-input-2382427777.py:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid = torch.tensor(valid_set, dtype=torch.float32).to(DEVICE)
/tmp/ipython-input-2382427777.py:28: UserWarning: To copy construct from a tensor, it is recommended to use so

Epoch 10, Loss: 0.4183
Epoch 20, Loss: 0.3744
Epoch 30, Loss: 0.3551
Epoch 40, Loss: 0.3384
Epoch 50, Loss: 0.3329
Epoch 60, Loss: 0.3316
Epoch 70, Loss: 0.3265
overfitting 74 0.3236457407474518 0.884
Predicted class: tensor([0, 3, 9,  ..., 3, 6, 3])
Epoch 10, Loss: 0.4229
Epoch 20, Loss: 0.3719
Epoch 30, Loss: 0.3497
Epoch 40, Loss: 0.3393
Epoch 50, Loss: 0.3304
Epoch 60, Loss: 0.3258
Epoch 70, Loss: 0.3237
Epoch 80, Loss: 0.3262
Epoch 90, Loss: 0.3198
Epoch 100, Loss: 0.3227
overfitting 99 0.32188311219215393 0.881
Predicted class: tensor([1, 1, 6,  ..., 6, 7, 0])
Epoch 10, Loss: 0.4246
Epoch 20, Loss: 0.3763
Epoch 30, Loss: 0.3502
Epoch 40, Loss: 0.3398
Epoch 50, Loss: 0.3311
overfitting 51 0.3114754557609558 0.8795714285714286
Predicted class: tensor([4, 6, 4,  ..., 9, 1, 4])


THIS NOTEBOOK WILL RUN FOR 5 OR 10% OF LABELS

In [26]:
#5% or 10% of data is unlabeled

if LABELED_DATA_PERCENT ==5 or LABELED_DATA_PERCENT ==10:
  #SPLIT DATA INTO SPECIFIED REGIONS
  test_set = X[0:7000] #10% of df, 7k data points
  y_test = Y[0:7000]

  valid_set = X[7000:14000] #10% of df, 7k data points
  y_valid = Y[7000:14000]

  train_set = X[14000:] #80% of df, 56k data points
  y_train = Y[14000:]

  #split training again for labeled/unlabeled
  unlabelled_train_set = train_set[0:53200] if LABELED_DATA_PERCENT == 5 else train_set[0:50400] #95% unlabelled or 90% unlabeled
  y_unlabelled_train = y_train[0:53200] if LABELED_DATA_PERCENT == 5 else y_train[0:50400] #95% unlabelled or 90% unlabeled

  labelled_train_set = train_set[53200:] if LABELED_DATA_PERCENT == 5 else train_set[50400:] #95% unlabelled or 90% unlabeled
  y_labelled_train = y_train[53200:] if LABELED_DATA_PERCENT == 5 else y_train[50400:] #95% unlabelled or 90% unlabeled

  unlabelled_train_set = unlabelled_train_set.cpu().detach().numpy()
  labelled_train_set = labelled_train_set.cpu().detach().numpy()
  y_labelled_train = y_labelled_train.cpu().detach().numpy()

  NUM_LABELS = labelled_train_set.shape[0] #for 5% and 10% data this will work
  NUM_ITERS = 16 if LABELED_DATA_PERCENT == 5 else 8
  #label prop

  #Since these take quite some time the prints help gauge progress
  new_label_indices = None #will store the pseudolabels that we say are good enough to be treated as real labels
  new_y = None
  for subset_split in range(NUM_ITERS): #we need exactly 4 subsets of the same size to label all the data
    print(f"starting {subset_split+1}th subset")
    subset = np.concatenate((labelled_train_set,unlabelled_train_set[NUM_LABELS*(subset_split):NUM_LABELS*(subset_split+1)]),axis=0)
    X = subset.reshape(subset.shape[0],784,1).squeeze()
    X = X.astype('float32')/255 #normalize and we are using float32 to save space
    subset = None # to save space

    T = np.zeros((NUM_LABELS*2,NUM_LABELS*2))
    #set up Y
    y =  np.eye(10)[y_labelled_train] #one hot encoding of the labelled dat
    y = np.concatenate((y,np.zeros((NUM_LABELS,10))),axis=0) #concat the unpredicted labels

    #this way of computing euclidean distance is from "Euclidean Distance Trick" referred in report
    row_sums = np.sum(X**2, axis=1).reshape(-1, 1)
    dists = row_sums - 2 *np.dot(X, X.T)+ row_sums.T

    T = np.exp(-dists / (2 * sigma**2)) #kernel

    print(f"training on the {subset_split+1}th subset")
    #training loop
    static_labels = np.eye(10)[y_labelled_train]
    for i in range(100):
      #propagate the labels
      y = T @ y

      #renormalize y, basically make sure each of the rows represent a valid probability distribution
      y = y / np.sum(y, axis=1, keepdims=True)

      #reset known labels
      y[0:NUM_LABELS] = static_labels
    print("done")
    #AFTER TRAINING LOOP - we will now evaulate the results by throwing any labels away that the model isnt super confiden  about

    #y_unlabelled_train = y_unlabelled_train.cpu().detach().numpy()
    temp = np.eye(10)[y_unlabelled_train[NUM_LABELS*(subset_split):NUM_LABELS*(subset_split+1)]]

    #filtering out the bad labels. just for fun, we will throw in the actual labels for the unlabeled data to see how accurate the predicted labels are.
    #note that we aren't using the unlabeled labels for the actual model, only for the comparison
    data_dict = {
        "unlabeled labels": np.argmax(temp,axis=1),
        "predicted labels": np.argmax(y[NUM_LABELS:],axis=1),
        "soft max" : np.max(y[NUM_LABELS:],axis=1)
    }
    df = pd.DataFrame(data_dict)
    before = len( df[df['predicted labels'] == df['unlabeled labels']])
    print("accuracy before", before/len(df))
    filter_df = df[df['soft max'] > 0.99]
    after = len(filter_df[filter_df['predicted labels'] == filter_df['unlabeled labels']])
    print("accuracy after",after/len(filter_df))

    if new_label_indices is None:
      new_label_indices = filter_df.index.to_numpy()
      new_y = filter_df['predicted labels'].to_numpy()
    else:
      new_label_indices = np.concatenate((new_label_indices,filter_df.index.to_numpy()+(subset_split*NUM_LABELS)))
      new_y = np.concatenate((new_y,filter_df['predicted labels'].to_numpy()))



starting 1th subset
training on the 1th subset
done
accuracy before 0.8058928571428572
accuracy after 0.9772612430520465
starting 2th subset
training on the 2th subset
done
accuracy before 0.8169642857142857
accuracy after 0.979313824419778
starting 3th subset
training on the 3th subset
done
accuracy before 0.8080357142857143
accuracy after 0.9755381604696673
starting 4th subset
training on the 4th subset
done
accuracy before 0.82125
accuracy after 0.9832098765432099
starting 5th subset
training on the 5th subset
done
accuracy before 0.8132142857142857
accuracy after 0.9758501724987678
starting 6th subset
training on the 6th subset
done
accuracy before 0.8183928571428571
accuracy after 0.9802858551010349
starting 7th subset
training on the 7th subset
done
accuracy before 0.8201785714285714
accuracy after 0.9770573566084788
starting 8th subset
training on the 8th subset
done
accuracy before 0.8082142857142857
accuracy after 0.9771485345255837


In [27]:
if LABELED_DATA_PERCENT ==5 or LABELED_DATA_PERCENT ==10:
  new_train_set = unlabelled_train_set.reshape(unlabelled_train_set.shape[0],784,1).squeeze()
  new_train_set = new_train_set[new_label_indices]

  backup = new_train_set.copy()
  y_back = new_y.copy()
  X = torch.cat([X_train_raw, X_test_raw], dim=0) #do this again because we overriden X

  #conv to tensors
  np.random.seed(1)
  torch.manual_seed(1)

  #RANDOMLY PERMUTATE DATASET
  indices = np.random.permutation(len(Y))
  X = X[indices]
  Y = Y[indices]

  #SPLIT DATA INTO SPECIFIED REGIONS
  test_set = X[0:7000] #10% of df, 7k data points
  y_test = Y[0:7000]

  valid_set = X[7000:14000] #10% of df, 7k data points
  y_valid = Y[7000:14000]

  #assemble our new training set, and add the original labeled set back
  new_train_set = new_train_set.reshape(new_train_set.shape[0],28,28)
  new_train_set = np.concatenate((new_train_set,labelled_train_set))
  new_y = np.concatenate((new_y,y_labelled_train))

  new_train_set = torch.tensor(new_train_set, dtype=torch.float32).to(DEVICE)
  new_y = torch.tensor(new_y, dtype=torch.long).to(DEVICE)

  test = torch.tensor(test_set, dtype=torch.float32).to(DEVICE)
  y_test = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
  valid = torch.tensor(valid_set, dtype=torch.float32).to(DEVICE)
  y_valid = torch.tensor(y_valid, dtype=torch.long).to(DEVICE)

  for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)

    #INSTANTIATE MODEL
    model = NeuralNetwork().to(DEVICE)
    model.apply(weights_init)

    #DEFINE CRITERIA AND OPTIMIZER
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE ,momentum = MOMENTUM) #m = 0.5,0.99; lr = 0.001, 0.1
    #optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = None
    if ALR == "exp_decay":
      scheduler = ExponentialLR(optimizer, gamma=GAMMA)
    elif ALR == "factor_decay":
      scheduler = StepLR(optimizer, step_size=25, gamma=0.1)

    #NORMALIZE VALID AND TEST HERE B/C THEY AREN'T BEING AUGMENTED
    if NORMALIZATION == "minmax":
        test_ = test / 255.0 #TODO: REMOVE THIS FROM THE PREV CODE
        valid = valid / 255.0
        #new_train_set = new_train_set /255.0
    elif NORMALIZATION == "zscore":
        # Simple Z-score approximation
        valid = (valid - valid.mean()) / valid.std()
        test = (test - test.mean()) / test.std()
        #new_train_set = (new_train_set - new_train_set.mean()) / new_train_set.std()

    #DEFINE VARS FOR MINIBATCH GRADIENT DESCENT
    dataset = TensorDataset(new_train_set, new_y)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    #DATA COLLECTION FOR THIS SEED
    seed_train_loss, seed_valid_loss = [], []
    seed_train_acc, seed_valid_acc = [], []
    loss_history = np.zeros(OVERFITTING_DETECTION_PARAMETER)
    loss_history[loss_history == 0.0] = np.nan #fix this later

    #FORWARD PASS WITH INITIALIZED WEIGHTS
    for epoch in range(EPOCHS):
      model.train()

      #VARS FOR MINIBATCH STORAGE
      running_mean_loss = 0 #stores average loss over minibatches, so this is mean per epoch
      running_mean_acc = 0 #stores acc loss over minibatches, so this is acc per epoch
      count = 1 #counts minibatches

      #loads minibatch
      for x,y in dataloader:
          x, y = x.to(DEVICE), y.to(DEVICE)

          x = augmenter(x)
          # plt.imshow(x[0])
          # plt.show()

          if NORMALIZATION == "minmax":
              x = x / 255.0
          elif NORMALIZATION == "zscore":
              x = (x - x.mean()) / x.std()

          #forward pass and loss
          outputs = model(x)
          loss = criterion(outputs, y)

          #backprop
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          #get the loss and acc of the current minibatch
          minibatch_loss, minibatch_acc = eval_model(x, y,model,criterion)

          #store average loss and acc of minibatches
          running_mean_loss = running_mean_loss + (minibatch_loss - running_mean_loss) / count
          running_mean_acc = running_mean_acc + (minibatch_acc - running_mean_acc) / count
          count += 1

      #store the minibatch averages as per-epoch averages
      seed_train_loss.append(running_mean_loss)
      seed_train_acc.append(running_mean_acc)

      #validation
      model.eval()
      valid_loss, valid_acc = eval_model(valid, y_valid,model,criterion)

      seed_valid_loss.append(valid_loss)
      seed_valid_acc.append(valid_acc)

        #display avg. training loss
      if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}, Loss: {running_mean_loss:.4f}')
        print(f'Acc: {valid_acc:.4f}')

      loss_history[epoch%OVERFITTING_DETECTION_PARAMETER] = valid_loss

      #check overfitting
      mean = np.nanmean(loss_history)
      std = np.nanstd(loss_history)
      if valid_loss > mean + std or epoch == EPOCHS-1:
        print('overfitting',epoch,valid_loss,valid_acc)
        results['converge_time'].append(epoch)
        break

      if ALR == "exp_decay" or ALR == "factor_decay":
        scheduler.step() #decay learning rate
        # current_lr = optimizer.param_groups[0]['lr']
        # print(f"Current Learning Rate: {current_lr}")

    results['train_loss'].append(seed_train_loss)
    results['valid_loss'].append(seed_valid_loss)
    results["train_acc"].append(seed_train_acc)
    results["valid_acc"].append(seed_valid_acc)

    #TESTING
    logits = model(test)
    pred_probab = torch.nn.Softmax(dim=1)(logits)
    y_pred = pred_probab.argmax(1)
    print(f"Predicted class: {y_pred}")

    _,final_acc = eval_model(test,y_test,model,criterion)

    results["test_acc"].append(final_acc)


/tmp/ipython-input-2074090754.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test = torch.tensor(test_set, dtype=torch.float32).to(DEVICE)
/tmp/ipython-input-2074090754.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
/tmp/ipython-input-2074090754.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid = torch.tensor(valid_set, dtype=torch.float32).to(DEVICE)
/tmp/ipython-input-2074090754.py:36: UserWarning: To copy construct from a tensor, it is recommended to use so

Epoch 10, Loss: 0.2090
Acc: 0.7896
Epoch 20, Loss: 0.1751
Acc: 0.8083
Epoch 30, Loss: 0.1632
Acc: 0.8164
overfitting 38 0.5115699172019958 0.8191428571428572
Predicted class: tensor([0, 3, 9,  ..., 3, 2, 3])
Epoch 10, Loss: 0.2101
Acc: 0.7743
Epoch 20, Loss: 0.1767
Acc: 0.8049
Epoch 30, Loss: 0.1681
Acc: 0.8203
Epoch 40, Loss: 0.1588
Acc: 0.8246
Epoch 50, Loss: 0.1540
Acc: 0.8269
overfitting 57 0.4910198450088501 0.8264285714285714
Predicted class: tensor([0, 3, 9,  ..., 3, 2, 3])
Epoch 10, Loss: 0.2106
Acc: 0.7846
Epoch 20, Loss: 0.1790
Acc: 0.8101
Epoch 30, Loss: 0.1618
Acc: 0.8147
Epoch 40, Loss: 0.1568
Acc: 0.8223
overfitting 42 0.5015034675598145 0.8217142857142857
Predicted class: tensor([0, 3, 9,  ..., 3, 2, 3])


IF LABELED DATA PERCENT IS 50% OR 25%

In [106]:
#for this much data we will stick to the original 11200x11200 and instead take random samples of labeled
if LABELED_DATA_PERCENT ==50 or LABELED_DATA_PERCENT ==25:
  #SPLIT DATA INTO SPECIFIED REGIONS
  test_set = X[0:7000] #10% of df, 7k data points
  y_test = Y[0:7000]

  valid_set = X[7000:14000] #10% of df, 7k data points
  y_valid = Y[7000:14000]

  train_set = X[14000:] #80% of df, 56k data points
  y_train = Y[14000:]

  #split training again for labeled/unlabeled
  unlabelled_train_set = train_set[0:42000] if LABELED_DATA_PERCENT == 75 else train_set[0:28000]  #75% unlabelled or 50% unlabeled
  y_unlabelled_train = y_train[0:42000] if LABELED_DATA_PERCENT == 75 else y_train[0:28000]

  labelled_train_set = train_set[42000:] if LABELED_DATA_PERCENT == 75 else train_set[28000:] #25% labelled
  y_labelled_train = y_train[42000:] if LABELED_DATA_PERCENT == 75 else y_train[28000:]

  unlabelled_train_set = unlabelled_train_set.cpu().detach().numpy()
  labelled_train_set = labelled_train_set.cpu().detach().numpy()
  y_labelled_train = y_labelled_train.cpu().detach().numpy()

  NUM_LABELS = 7000 #divisble by both 25 and 50 percent
  NUM_ITERS = 6 if LABELED_DATA_PERCENT == 75 else 3 #this should be 3 for 50% b/c you'd go from 7k -> 28k

  #label prop

  #Since these take quite some time the prints help gauge progress
  new_label_indices = None #will store the pseudolabels that we say are good enough to be treated as real labels
  new_y = None
  for subset_split in range(NUM_ITERS): #we need exactly 4 subsets of the same size to label all the data
    print(f"starting {subset_split+1}th subset")

    #we have to take subset of the labeled training now
    indices = np.random.default_rng(seed=subset_split).integers(low=0, high=14000, size=7000)
    labeled_subset = labelled_train_set[indices]
    y_labeled_subset = y_labelled_train[indices]

    subset = np.concatenate((labeled_subset,unlabelled_train_set[NUM_LABELS*(subset_split):NUM_LABELS*(subset_split+1)]),axis=0)
    X = subset.reshape(subset.shape[0],784,1).squeeze()
    X = X.astype('float32')/255 #normalize and we are using float32 to save space
    subset = None # to save space

    T = np.zeros((NUM_LABELS*2,NUM_LABELS*2))
    #set up Y
    y =  np.eye(10)[y_labeled_subset] #one hot encoding of the labelled dat
    y = np.concatenate((y,np.zeros((NUM_LABELS,10))),axis=0) #concat the unpredicted labels

    #this way of computing euclidean distance is from "Euclidean Distance Trick" referred in report
    row_sums = np.sum(X**2, axis=1).reshape(-1, 1)
    dists = row_sums - 2 *np.dot(X, X.T)+ row_sums.T

    T = np.exp(-dists / (2 * sigma**2)) #kernel

    print(f"training on the {subset_split+1}th subset")
    #training loop
    static_labels = np.eye(10)[y_labeled_subset]
    for i in range(100):
      #propagate the labels
      y = T @ y

      #renormalize y, basically make sure each of the rows represent a valid probability distribution
      y = y / np.sum(y, axis=1, keepdims=True)

      #reset known labels
      y[0:NUM_LABELS] = static_labels
    print("done")
    #AFTER TRAINING LOOP - we will now evaulate the results by throwing any labels away that the model isnt super confiden  about

    #y_unlabelled_train = y_unlabelled_train.cpu().detach().numpy()
    temp = np.eye(10)[y_unlabelled_train[NUM_LABELS*(subset_split):NUM_LABELS*(subset_split+1)]]

    #filtering out the bad labels. just for fun, we will throw in the actual labels for the unlabeled data to see how accurate the predicted labels are.
    #note that we aren't using the unlabeled labels for the actual model, only for the comparison
    data_dict = {
        "unlabeled labels": np.argmax(temp,axis=1),
        "predicted labels": np.argmax(y[NUM_LABELS:],axis=1),
        "soft max" : np.max(y[NUM_LABELS:],axis=1)
    }
    df = pd.DataFrame(data_dict)
    before = len( df[df['predicted labels'] == df['unlabeled labels']])
    print("accuracy before", before/len(df))
    filter_df = df[df['soft max'] > 0.99]
    after = len(filter_df[filter_df['predicted labels'] == filter_df['unlabeled labels']])
    print("accuracy after",after/len(filter_df))

    if new_label_indices is None:
      new_label_indices = filter_df.index.to_numpy()
      new_y = filter_df['predicted labels'].to_numpy()
    else:
      new_label_indices = np.concatenate((new_label_indices,filter_df.index.to_numpy()+(subset_split*NUM_LABELS)))
      new_y = np.concatenate((new_y,filter_df['predicted labels'].to_numpy()))



starting 1th subset
training on the 1th subset
done
accuracy before 0.801
accuracy after 0.9734848484848485
starting 2th subset
training on the 2th subset
done
accuracy before 0.811
accuracy after 0.9754768392370572
starting 3th subset
training on the 3th subset
done
accuracy before 0.813
accuracy after 0.9732680722891566


In [107]:
if LABELED_DATA_PERCENT ==25 or LABELED_DATA_PERCENT ==50:
  new_train_set = unlabelled_train_set.reshape(unlabelled_train_set.shape[0],784,1).squeeze()
  new_train_set = new_train_set[new_label_indices]

  backup = new_train_set.copy()
  y_back = new_y.copy()
  X = torch.cat([X_train_raw, X_test_raw], dim=0) #do this again because we overriden X

  #conv to tensors
  np.random.seed(1)
  torch.manual_seed(1)

  #RANDOMLY PERMUTATE DATASET
  indices = np.random.permutation(len(Y))
  X = X[indices]
  Y = Y[indices]

  #SPLIT DATA INTO SPECIFIED REGIONS
  test_set = X[0:7000] #10% of df, 7k data points
  y_test = Y[0:7000]

  valid_set = X[7000:14000] #10% of df, 7k data points
  y_valid = Y[7000:14000]

  #assemble our new training set, and add the original labeled set back
  new_train_set = new_train_set.reshape(new_train_set.shape[0],28,28)
  new_train_set = np.concatenate((new_train_set,labelled_train_set))
  new_y = np.concatenate((new_y,y_labelled_train))

  new_train_set = torch.tensor(new_train_set, dtype=torch.float32).to(DEVICE)
  new_y = torch.tensor(new_y, dtype=torch.long).to(DEVICE)

  test = torch.tensor(test_set, dtype=torch.float32).to(DEVICE)
  y_test = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
  valid = torch.tensor(valid_set, dtype=torch.float32).to(DEVICE)
  y_valid = torch.tensor(y_valid, dtype=torch.long).to(DEVICE)

  for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)

    #INSTANTIATE MODEL
    model = NeuralNetwork().to(DEVICE)
    model.apply(weights_init)

    #DEFINE CRITERIA AND OPTIMIZER
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE ,momentum = MOMENTUM) #m = 0.5,0.99; lr = 0.001, 0.1
    #optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = None
    if ALR == "exp_decay":
      scheduler = ExponentialLR(optimizer, gamma=GAMMA)
    elif ALR == "factor_decay":
      scheduler = StepLR(optimizer, step_size=25, gamma=0.1)

    #NORMALIZE VALID AND TEST HERE B/C THEY AREN'T BEING AUGMENTED
    if NORMALIZATION == "minmax":
        test_ = test / 255.0 #TODO: REMOVE THIS FROM THE PREV CODE
        valid = valid / 255.0
        #new_train_set = new_train_set /255.0
    elif NORMALIZATION == "zscore":
        # Simple Z-score approximation
        valid = (valid - valid.mean()) / valid.std()
        test = (test - test.mean()) / test.std()
        #new_train_set = (new_train_set - new_train_set.mean()) / new_train_set.std()

    #DEFINE VARS FOR MINIBATCH GRADIENT DESCENT
    dataset = TensorDataset(new_train_set, new_y)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    #DATA COLLECTION FOR THIS SEED
    seed_train_loss, seed_valid_loss = [], []
    seed_train_acc, seed_valid_acc = [], []
    loss_history = np.zeros(OVERFITTING_DETECTION_PARAMETER)
    loss_history[loss_history == 0.0] = np.nan #fix this later

    #FORWARD PASS WITH INITIALIZED WEIGHTS
    for epoch in range(EPOCHS):
      model.train()

      #VARS FOR MINIBATCH STORAGE
      running_mean_loss = 0 #stores average loss over minibatches, so this is mean per epoch
      running_mean_acc = 0 #stores acc loss over minibatches, so this is acc per epoch
      count = 1 #counts minibatches

      #loads minibatch
      for x,y in dataloader:
          x, y = x.to(DEVICE), y.to(DEVICE)

          x = augmenter(x)
          # plt.imshow(x[0])
          # plt.show()

          if NORMALIZATION == "minmax":
              x = x / 255.0
          elif NORMALIZATION == "zscore":
              x = (x - x.mean()) / x.std()

          #forward pass and loss
          outputs = model(x)
          loss = criterion(outputs, y)

          #backprop
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          #get the loss and acc of the current minibatch
          minibatch_loss, minibatch_acc = eval_model(x, y,model,criterion)

          #store average loss and acc of minibatches
          running_mean_loss = running_mean_loss + (minibatch_loss - running_mean_loss) / count
          running_mean_acc = running_mean_acc + (minibatch_acc - running_mean_acc) / count
          count += 1

      #store the minibatch averages as per-epoch averages
      seed_train_loss.append(running_mean_loss)
      seed_train_acc.append(running_mean_acc)

      #validation
      model.eval()
      valid_loss, valid_acc = eval_model(valid, y_valid,model,criterion)

      seed_valid_loss.append(valid_loss)
      seed_valid_acc.append(valid_acc)

        #display avg. training loss
      if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}, Loss: {running_mean_loss:.4f}')
        print(f'Acc: {valid_acc:.4f}')

      loss_history[epoch%OVERFITTING_DETECTION_PARAMETER] = valid_loss

      #check overfitting
      mean = np.nanmean(loss_history)
      std = np.nanstd(loss_history)
      if valid_loss > mean + std or epoch == EPOCHS-1:
        print('overfitting',epoch,valid_loss,valid_acc)
        results['converge_time'].append(epoch)
        break

      if ALR == "exp_decay" or ALR == "factor_decay":
        scheduler.step() #decay learning rate
        # current_lr = optimizer.param_groups[0]['lr']
        # print(f"Current Learning Rate: {current_lr}")

    results['train_loss'].append(seed_train_loss)
    results['valid_loss'].append(seed_valid_loss)
    results["train_acc"].append(seed_train_acc)
    results["valid_acc"].append(seed_valid_acc)

    #TESTING
    logits = model(test)
    pred_probab = torch.nn.Softmax(dim=1)(logits)
    y_pred = pred_probab.argmax(1)
    print(f"Predicted class: {y_pred}")

    _,final_acc = eval_model(test,y_test,model,criterion)

    results["test_acc"].append(final_acc)


/tmp/ipython-input-3523859831.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test = torch.tensor(test_set, dtype=torch.float32).to(DEVICE)
/tmp/ipython-input-3523859831.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
/tmp/ipython-input-3523859831.py:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid = torch.tensor(valid_set, dtype=torch.float32).to(DEVICE)
/tmp/ipython-input-3523859831.py:36: UserWarning: To copy construct from a tensor, it is recommended to use so

Epoch 10, Loss: 0.3985
Acc: 0.8337
Epoch 20, Loss: 0.3464
Acc: 0.8450
Epoch 30, Loss: 0.3291
Acc: 0.8597
overfitting 38 0.3783123195171356 0.8635714285714285
Predicted class: tensor([0, 3, 9,  ..., 1, 0, 3])
Epoch 10, Loss: 0.4387
Acc: 0.8090
Epoch 20, Loss: 0.3607
Acc: 0.8467
Epoch 30, Loss: 0.3430
Acc: 0.8611
Epoch 40, Loss: 0.3196
Acc: 0.8597
Epoch 50, Loss: 0.3120
Acc: 0.8681
Epoch 60, Loss: 0.3062
Acc: 0.8719
Epoch 70, Loss: 0.3088
Acc: 0.8713
overfitting 70 0.3545128405094147 0.8694285714285714
Predicted class: tensor([0, 3, 9,  ..., 3, 0, 0])
Epoch 10, Loss: 0.3953
Acc: 0.8391
Epoch 20, Loss: 0.3819
Acc: 0.8424
Epoch 30, Loss: 0.3276
Acc: 0.8513
Epoch 40, Loss: 0.3241
Acc: 0.8681
Epoch 50, Loss: 0.3100
Acc: 0.8723
Epoch 60, Loss: 0.3048
Acc: 0.8714
Epoch 70, Loss: 0.3018
Acc: 0.8723
overfitting 71 0.34880825877189636 0.8712857142857143
Predicted class: tensor([0, 3, 9,  ..., 3, 2, 3])


In [109]:
np.mean([results['test_acc']])

np.float64(0.8732857142857142)

In [119]:
def pad_results(results,convergence_list):
  max_ = max(convergence_list) +1 #this is the value we extend the other lists to

  for i in range(len(results["train_loss"])):
    get_max = results['converge_time'][i]+1
    #if the current seed isnt the one with the longest iterations, pad it to reach the max iterations

    while get_max < max_:
      results["valid_acc"][i].append(results["valid_acc"][i][-1])
      results["train_loss"][i].append(results["train_loss"][i][-1])
      results["valid_loss"][i].append(results["valid_loss"][i][-1])
      results["train_acc"][i].append(results["train_acc"][i][-1])
      get_max+=1

pad_results(results,results["converge_time"])

In [120]:

#create dataframe

data_dict = {'training_loss': np.mean(np.array(results["train_loss"]),axis=0), 'validation_loss': np.mean(np.array(results["valid_loss"]),axis=0), 'training_accuracy': np.mean(np.array(results["train_acc"]),axis=0), 'validation_accuracy': np.mean(np.array(results["valid_acc"]),axis=0)}
pd.DataFrame(data_dict)

,training_loss,validation_loss,training_accuracy,validation_accuracy
0,1.146528,0.715590,0.566106,0.725952
1,0.583538,0.700763,0.781130,0.733429
2,0.540915,0.586657,0.797274,0.775762
3,0.490307,0.537765,0.815845,0.796857
4,0.458873,0.534862,0.829655,0.801000
...,...,...,...,...
67,0.309421,0.359854,0.886555,0.868333
68,0.308888,0.359583,0.886499,0.868810
69,0.309506,0.358235,0.886555,0.869048
70,0.310678,0.360282,0.885798,0.868048


In [121]:
pd.DataFrame(data_dict).to_csv('stage5_50_labeled.csv')

In [133]:
import numpy as np
from scipy import stats

stage1 = [0.0991,0.0997,0.0973]
stage2 = [0.7934,0.7641,0.819]
stage3 = [0.8316,0.8191,0.8333]
dropout = [0.8369,0.8089,0.8218]
decay = [0.8413,0.8251,0.8407]
ssl = [0.8450,0.8486,0.8327]

t_stat, p_val = stats.ttest_ind(ssl, stage2)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")

if p_val < 0.05:
    print("statistically significant")

t-statistic: 3.0126
p-value: 0.0394
statistically significant


In [136]:
np.std([0.8450,0.8486,0.8327])

np.float64(0.006807348970047008)